In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
if os.path.exists("/content/commitgen"):
    !cd /content/commitgen && git pull
else:
    !git clone https://github.com/Nilay-Mehta/commitgen.git /content/commitgen
%cd /content/commitgen

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
if not os.path.exists("data/test.jsonl"):
    !python data/prepare_dataset.py
else:
    print("data/test.jsonl already exists, skipping prepare_dataset")

In [ ]:
from pathlib import Path
from evaluation.evaluate import evaluate

TEST_JSONL = Path("/content/commitgen/data/test.jsonl")
OUTPUT_DIR = Path("/content/commitgen/evaluation/results")
ADAPTER_PATH = Path("/content/drive/MyDrive/commitgen_checkpoints/qwen25coder-lora-full-0/checkpoint-2000")
DEVICE = "cpu"
# Set LIMIT to a small int (e.g. 20) for a quick smoke run. None = full 500.
LIMIT = None

In [ ]:
metrics_base = evaluate(
    test_jsonl=TEST_JSONL,
    output_dir=OUTPUT_DIR,
    adapter_path=None,
    label="base",
    device=DEVICE,
    limit=LIMIT,
)

In [ ]:
metrics_tuned = evaluate(
    test_jsonl=TEST_JSONL,
    output_dir=OUTPUT_DIR,
    adapter_path=ADAPTER_PATH,
    label="tuned",
    device=DEVICE,
    limit=LIMIT,
)

In [ ]:
print(f"{'Metric':<22} {'Base':>10} {'Tuned':>10} {'Delta':>10}")
print("-" * 56)
for key in ["bleu4", "rouge_l", "type_accuracy_pct", "length_ratio"]:
    b = metrics_base[key]
    t = metrics_tuned[key]
    print(f"{key:<22} {b:>10.2f} {t:>10.2f} {t-b:>+10.2f}")

In [ ]:
import json
import random

base_preds = [json.loads(l) for l in open(OUTPUT_DIR / "predictions_base.jsonl", encoding="utf-8")]
tuned_preds = [json.loads(l) for l in open(OUTPUT_DIR / "predictions_tuned.jsonl", encoding="utf-8")]
random.seed(42)
indices = random.sample(range(len(base_preds)), min(20, len(base_preds)))
for i in indices:
    print(f"--- example {i} ---")
    print(f"DIFF (truncated): {base_preds[i]['diff'][:200]}...")
    print(f"REF:   {base_preds[i]['reference']}")
    print(f"BASE:  {base_preds[i]['prediction']}")
    print(f"TUNED: {tuned_preds[i]['prediction']}")
    print()